# Resultados Compra Ágil desde SQLite preparado — diagnóstico conservador

[![Abrir en Google Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/juliopez/MercadoPublico-Inteligencia/blob/main/colab/Resultados_Compra_Agil_Colab.ipynb)

No vuelve a leer ZIP ni consulta API. Usa el SQLite guardado por el importador Compra Ágil y la hoja clonada `Resultados`. **Solo propone una fila por proceso cuando existe exactamente una oferta marcada explícitamente como seleccionada.** La fuente no equivale a adjudicación verificada: el campo `adjudicatario` solo representa al proveedor marcado como seleccionado y `validado` deja constancia de verificación pendiente. No infiere `fecha_resultado` ni `motivo_seleccion`.

**Atención:** el identificador de cada oferta es un hash sintético, no un código oficial. No se atribuye OC ganadora sin prueba documental. No escribe por defecto.

In [ ]:
!pip -q install gspread google-api-python-client google-auth
from google.colab import auth
from google.auth import default
from googleapiclient.discovery import build
from googleapiclient.http import MediaIoBaseDownload
import gspread,sqlite3,io,json,datetime,time,os,unicodedata,re
auth.authenticate_user()
creds,_=default(scopes=['https://www.googleapis.com/auth/spreadsheets','https://www.googleapis.com/auth/drive'])
gc=gspread.authorize(creds);drive=build('drive','v3',credentials=creds)
CLON='1loeK_IM3RXHaR7CqpjUbz-TsNgOKNrV1ejq8aRC6JyE'
HEAD=['codigo','adjudicatario','monto_adjudicado_clp','fecha_resultado','motivo_seleccion','fuente','validado','id_cotizacion_ganadora','monto_oferta_seleccionada_clp','estado_verificacion_oc']
ws=gc.open_by_key(CLON).worksheet('Resultados')
assert ws.row_values(1)==HEAD
print('Clon confirmado; hoja Resultados')

## Restaurar la preparación anterior

Pega el identificador `PREPARADO_ID` generado por `Compra_Agil_Procesos_Ofertas_Colab.ipynb`. Es un archivo SQLite de Drive; no requiere ZIP.

In [ ]:
PREPARADO_ID='' # pega aquí el ID de Drive generado al cargar COT_2026-08.zip
assert PREPARADO_ID.strip(),'Falta PREPARADO_ID: no se debe reprocesar ZIP'
ruta='/content/compra_agil_preparado_resultados.sqlite'
with open(ruta,'wb') as f:
 download=MediaIoBaseDownload(f,drive.files().get_media(fileId=PREPARADO_ID))
 listo=False
 while not listo: _,listo=download.next_chunk()
db=sqlite3.connect('file:'+ruta+'?mode=ro',uri=True)
assert db.execute('PRAGMA integrity_check').fetchone()[0]=='ok'
assert {'procesos','ofertas'}.issubset({x[0] for x in db.execute("SELECT name FROM sqlite_master WHERE type='table'")})
print('Preparación recuperada: ',round(os.path.getsize(ruta)/1e6,1),'MB')

## Informe de elegibilidad

Se considera seleccionada solo una marca explícita `sí/si/true/1/seleccionado`; cualquier marca no reconocida queda fuera, y los casos con múltiples seleccionadas por proceso se excluyen por ambigüedad. `monto_adjudicado_clp` queda vacío: el monto de la oferta seleccionada no acredita por sí solo monto adjudicado.

In [ ]:
def normal(x):
 x=unicodedata.normalize('NFKD',str(x or '').strip().lower())
 return ''.join(c for c in x if not unicodedata.combining(c))
MARCAS={'si','true','1','seleccionado','yes'}
seleccionadas={}
conteos={'ofertas_leidas':0,'marcadas_seleccionadas':0,'marcas_desconocidas':0}
for (raw,) in db.execute('SELECT fila FROM ofertas'):
 r=json.loads(raw);conteos['ofertas_leidas']+=1
 marca=normal(r[6])
 if marca not in {'si','no','true','false','1','0','yes','seleccionado',''}:conteos['marcas_desconocidas']+=1
 if marca in MARCAS:
  conteos['marcadas_seleccionadas']+=1
  seleccionadas.setdefault(r[0],[]).append(r)
existentes=set(ws.col_values(1)[1:])
candidatos=[];ambiguos=0;sin_proveedor=0
for codigo,rs in seleccionadas.items():
 if len(rs)!=1:ambiguos+=1;continue
 r=rs[0]
 if not(r[1].strip() and r[2].strip()):sin_proveedor+=1;continue
 if codigo in existentes:continue
 candidatos.append([codigo,r[1],'','','','Histórico Compra Ágil COT_2026-08.zip',
                    'Pendiente: selección histórica, no adjudicación verificada',r[5],r[2],'OC no verificada'])
reporte={**conteos,'procesos_con_marca_seleccionada':len(seleccionadas),
 'procesos_con_multiples_seleccionadas':ambiguos,'sin_datos_minimos':sin_proveedor,
 'procesos_ya_en_resultados':sum(1 for c in seleccionadas if c in existentes),
 'nuevos_candidatos':len(candidatos),
 'advertencia':'Proveedor y monto de oferta seleccionada; adjudicación y OC sin verificar'}
print(json.dumps(reporte,ensure_ascii=False,indent=2))
print('Primeras 5 filas candidatas:')
for r in candidatos[:5]:print(r)
with open('/content/reporte_resultados_compra_agil.json','w') as f:json.dump(reporte,f,indent=2,ensure_ascii=False)

## Escritura opcional — solo si el informe es coherente

Antes de cargar es indispensable revisar las marcas de selección y los casos ambiguos. Escribe la frase explícita en la siguiente celda para crear respaldo y cargar solo los candidatos. **No** convierte la selección histórica en adjudicación verificada.

In [ ]:
AUTORIZACION='' # AUTORIZO RESULTADOS PENDIENTES EN CLON
if AUTORIZACION!='AUTORIZO RESULTADOS PENDIENTES EN CLON':
 print('Simulación; no se modificó Google Sheets.')
else:
 assert CLON=='1loeK_IM3RXHaR7CqpjUbz-TsNgOKNrV1ejq8aRC6JyE'
 assert ws.row_values(1)==HEAD
 copia=drive.files().copy(fileId=CLON,body={'name':'Respaldo Resultados clon '+datetime.datetime.now().isoformat()},fields='id,webViewLink').execute()
 print('Respaldo:',copia.get('webViewLink',copia['id']))
 actual=[c.strip() for c in ws.col_values(1)[1:] if c.strip()]
 assert len(actual)==len(set(actual))
 presentes=set(actual)
 faltantes=[r for r in candidatos if r[0] not in presentes]
 requerido=len(presentes)+len(faltantes)+1
 if ws.row_count<requerido:ws.add_rows(requerido-ws.row_count)
 escritos=0
 for i in range(0,len(faltantes),250):
  lote=faltantes[i:i+250]
  inicio=len(presentes)+2
  rango=f'A{inicio}:J{inicio+len(lote)-1}'
  ws.update(range_name=rango,values=lote,value_input_option='RAW')
  confirmados={r[0] for r in ws.get(rango)}
  assert confirmados=={r[0] for r in lote},'Lote no confirmado: detener'
  presentes.update(confirmados);escritos+=len(lote)
  if escritos%5000==0 or escritos==len(faltantes):print('Resultados pendientes insertados:',escritos)
  time.sleep(0.5)
 print('CARGA FINALIZADA',escritos,'registros nuevos; total',len(presentes))